In [0]:
%python

# Databricks notebook source
# MAGIC %md
# MAGIC # Module 1 -- Business Intelligence Assistant
# MAGIC
# MAGIC Turns the BI layer's computed stats into an AI-generated executive summary and
# MAGIC recommendations. Lives in `ai/`, depends on `01_claude_api_setup.py` (same folder)
# MAGIC and reads `miya_academy.default.bi_summary_metrics` (written by
# MAGIC `bi/02_export_bi_summary.py`) -- it does not depend on the BI dashboard notebook
# MAGIC directly, so changes to that notebook's internals don't break this one.
# MAGIC
# MAGIC **The one rule this notebook follows strictly: Python computes every number, Claude
# MAGIC only narrates.** Claude is not asked to calculate anything -- it receives numbers
# MAGIC that are already computed and is instructed not to introduce new ones. A guard at
# MAGIC the end checks the response for numbers that don't trace back to the input, so a
# MAGIC hallucinated figure gets flagged instead of silently ending up on a slide.

# COMMAND ----------

%pip install "anthropic<0.45.0" --quiet

# COMMAND ----------

import anthropic
import time
import hashlib
import json

# Initialize Claude client
SECRET_SCOPE = "claude-api"
SECRET_KEY = "anthropic-key"
CLAUDE_MODEL = "claude-sonnet-5"

CLAUDE_API_KEY = dbutils.secrets.get(scope=SECRET_SCOPE, key=SECRET_KEY)
client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)

# Rate limiter
class RateLimiter:
    def __init__(self, min_interval_seconds: float = 1.0):
        self.min_interval = min_interval_seconds
        self._last_call = 0.0

    def wait(self):
        elapsed = time.time() - self._last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self._last_call = time.time()

rate_limiter = RateLimiter(min_interval_seconds=1.0)

# Response cache
_response_cache = {}

def _cache_key(prompt: str, model: str, **kwargs) -> str:
    payload = json.dumps({"prompt": prompt, "model": model, **kwargs}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()

def _call_with_retry(prompt: str, model: str = CLAUDE_MODEL, max_tokens: int = 1024,
                      max_retries: int = 3, **kwargs):
    last_error = None
    for attempt in range(max_retries):
        try:
            rate_limiter.wait()
            return client.messages.create(
                model=model,
                max_tokens=max_tokens,
                messages=[{"role": "user", "content": prompt}],
                **kwargs,
            )
        except anthropic.RateLimitError as e:
            wait_time = 2 ** attempt * 2
            print(f"Rate limited -- retrying in {wait_time}s (attempt {attempt + 1}/{max_retries})")
            time.sleep(wait_time)
            last_error = e
        except anthropic.APIStatusError as e:
            if e.status_code >= 500:
                wait_time = 2 ** attempt * 2
                print(f"Server error {e.status_code} -- retrying in {wait_time}s (attempt {attempt + 1}/{max_retries})")
                time.sleep(wait_time)
                last_error = e
            else:
                raise
    raise last_error

def call_claude(prompt: str, model: str = CLAUDE_MODEL, max_tokens: int = 1024, **kwargs):
    """Call Claude with caching + rate limiting + retry."""
    key = _cache_key(prompt, model, max_tokens=max_tokens, **kwargs)
    if key in _response_cache:
        return _response_cache[key], True
    response = _call_with_retry(prompt, model=model, max_tokens=max_tokens, **kwargs)
    text = response.content[0].text
    _response_cache[key] = text
    return text, False

print("Claude API setup complete")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Load the computed metrics
# MAGIC
# MAGIC No PySpark analysis happens in this notebook -- `bi/02_export_bi_summary.py` already
# MAGIC did that. This is just a read.

# COMMAND ----------

SUMMARY_TABLE = "miya_academy.default.bi_summary_metrics"

summary_row = spark.table(SUMMARY_TABLE).collect()[0]
stats = summary_row.asDict()

print("Loaded metrics:")
for k, v in stats.items():
    print(f"  {k}: {v}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## ROI assumptions -- explicit, optional, off by default
# MAGIC
# MAGIC This data is synthetic. Any dollar figure ("$X million retention problem") is only
# MAGIC as real as the assumptions behind it. Rather than let Claude invent a number, ROI
# MAGIC in dollars is only computed if you fill in real assumptions below. Leave them as
# MAGIC `None` and the summary reports impact in percentages and conversation counts only --
# MAGIC which is honest for a synthetic dataset and still useful.

# COMMAND ----------

# Fill these in with real, defensible numbers before enabling dollar-figure ROI.
# Leave as None to skip $ ROI and report relative impact only.
ASSUMPTIONS = {
    "avg_value_per_completed_conversation": None,  # e.g. 15.00 (currency units)
    "agent_cost_per_escalated_conversation": None,  # e.g. 3.50
}

roi_enabled = all(v is not None for v in ASSUMPTIONS.values())

if roi_enabled:
    recovered_conversations_if_half_dropoff_fixed = stats["early_dropoff_count"] * 0.5
    estimated_value_recovered = (
        recovered_conversations_if_half_dropoff_fixed * ASSUMPTIONS["avg_value_per_completed_conversation"]
    )
    print(f"ROI enabled. Illustrative recovered value if early drop-off is halved: "
          f"{estimated_value_recovered:,.2f} (using stated assumptions)")
else:
    estimated_value_recovered = None
    print("ROI assumptions not set -- report will use percentages/counts only, no dollar figures.")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Build the prompt
# MAGIC
# MAGIC The prompt hands Claude the numbers as structured data and explicitly forbids it
# MAGIC from computing or introducing new figures. This is the guardrail that matters most --
# MAGIC everything else here is secondary to this one instruction.

# COMMAND ----------

import json

context_json = json.dumps(stats, indent=2, default=str)

roi_instruction = (
    f"An illustrative recovered-value estimate has been computed: {estimated_value_recovered:,.2f} "
    f"(if early drop-off is halved, using the stated assumptions). You may reference this exact "
    f"figure once, clearly labeled as illustrative."
    if roi_enabled else
    "No dollar-value ROI has been computed (assumptions not set) -- do not invent a dollar figure. "
    "Discuss impact only in terms of the percentages and conversation counts provided below."
)

prompt = f"""You are a business intelligence analyst reviewing chatbot analytics for a WhatsApp
customer service bot. All data below is SYNTHETIC (generated for a portfolio project), not real
customer data -- do not refer to it as real production data.

METRICS (the only numbers you may use -- do not calculate, estimate, or introduce any other numbers):
{context_json}

{roi_instruction}

Write a business intelligence summary with these sections:
1. Executive Summary (3-4 sentences, plain language, for a non-technical stakeholder)
2. Key Findings (3-5 bullet points, each referencing a specific metric above by its exact value)
3. Root Cause Analysis (why might these patterns exist, based on the metrics -- clearly label this
   as interpretation/hypothesis, not fact)
4. Recommendations (3-4 concrete, prioritized actions)

Rules:
- Every number you state must appear in the METRICS block above, or be the single illustrative
  ROI figure explicitly provided (if any). Do not perform arithmetic on the numbers given --
  if a derived number would help, describe the relationship in words instead
  (e.g. "roughly 1 in 5" rather than computing a new percentage).
- Do not claim this is based on real customer data.
"""

# COMMAND ----------

# MAGIC %md
# MAGIC ## Call Claude and check the response for hallucinated numbers
# MAGIC
# MAGIC Cheap guard, not a proof of correctness: extract every number in the response and flag
# MAGIC any that don't appear in the input stats (or the ROI figure). It won't catch a
# MAGIC hallucinated number that happens to coincide with a real one, but it catches the
# MAGIC common case -- Claude inventing a plausible-looking statistic.

# COMMAND ----------

import re

response_text, was_cached = call_claude(prompt, max_tokens=1500)
print(f"(cache hit: {was_cached})\n")
print(response_text)

# COMMAND ----------

def extract_numbers(text):
    return set(re.findall(r"\d[\d,]*\.?\d*", text))

def normalize(n):
    return n.replace(",", "").rstrip(".")

allowed_numbers = set()
for v in stats.values():
    allowed_numbers.add(normalize(str(v)))
if estimated_value_recovered is not None:
    allowed_numbers.add(normalize(f"{estimated_value_recovered:.2f}"))
    allowed_numbers.add(normalize(f"{estimated_value_recovered:,.2f}"))

response_numbers = {normalize(n) for n in extract_numbers(response_text)}

# Small numbers (section headers like "1.", "2.", years, etc.) cause noise -- only flag
# numbers of a length/shape unlikely to be a list marker or stray digit.
suspicious = [
    n for n in response_numbers
    if n not in allowed_numbers and len(n.replace(".", "")) >= 2
]

print("\n" + "=" * 70)
if suspicious:
    print(f"REVIEW NEEDED -- {len(suspicious)} number(s) in the response don't trace back to the input stats:")
    for n in suspicious:
        print(f"  - {n}")
    print("These may be hallucinated. Check the response above before using it anywhere external.")
else:
    print("All numbers in the response trace back to the input metrics. No hallucination flags.")
print("=" * 70)
